In [1]:
#****************************************************************************
# (C) Cloudera, Inc. 2020-2023
#  All rights reserved.
#
#  Applicable Open Source License: GNU Affero General Public License v3.0
#
#  NOTE: Cloudera open source products are modular software products
#  made up of hundreds of individual components, each of which was
#  individually copyrighted.  Each Cloudera open source product is a
#  collective work under U.S. Copyright Law. Your license to use the
#  collective work is as provided in your written agreement with
#  Cloudera.  Used apart from the collective work, this file is
#  licensed for your use pursuant to the open source license
#  identified above.
#
#  This code is provided to you pursuant a written agreement with
#  (i) Cloudera, Inc. or (ii) a third-party authorized to distribute
#  this code. If you do not have a written agreement with Cloudera nor
#  with an authorized and properly licensed third party, you do not
#  have any rights to access nor to use this code.
#
#  Absent a written agreement with Cloudera, Inc. (“Cloudera”) to the
#  contrary, A) CLOUDERA PROVIDES THIS CODE TO YOU WITHOUT WARRANTIES OF ANY
#  KIND; (B) CLOUDERA DISCLAIMS ANY AND ALL EXPRESS AND IMPLIED
#  WARRANTIES WITH RESPECT TO THIS CODE, INCLUDING BUT NOT LIMITED TO
#  IMPLIED WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY AND
#  FITNESS FOR A PARTICULAR PURPOSE; (C) CLOUDERA IS NOT LIABLE TO YOU,
#  AND WILL NOT DEFEND, INDEMNIFY, NOR HOLD YOU HARMLESS FOR ANY CLAIMS
#  ARISING FROM OR RELATED TO THE CODE; AND (D)WITH RESPECT TO YOUR EXERCISE
#  OF ANY RIGHTS GRANTED TO YOU FOR THE CODE, CLOUDERA IS NOT LIABLE FOR ANY
#  DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, PUNITIVE OR
#  CONSEQUENTIAL DAMAGES INCLUDING, BUT NOT LIMITED TO, DAMAGES
#  RELATED TO LOST REVENUE, LOST PROFITS, LOSS OF INCOME, LOSS OF
#  BUSINESS ADVANTAGE OR UNAVAILABILITY, OR LOSS OR CORRUPTION OF
#  DATA.
#
# #  Author(s): Paul de Fusco
#***************************************************************************/

In [ ]:
import os
import sys
from datetime import date

import mlflow
import mlflow.onnx
import numpy as np
import onnxmltools
import pandas as pd
from mlflow.models import infer_signature
from onnxmltools.convert.common.data_types import FloatTensorType
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Notebook lives in xgboost/; make db.py and pii_datagen.py (repo root)
# importable without installing the project as a package.
sys.path.insert(0, os.path.abspath(".."))
from db import DB_PATH, get_conn, init_schema, seed_offers  # noqa: E402
from pii_datagen import _EMP_STATUSES, seed_customers  # noqa: E402

In [ ]:
import xgboost

print(f"MLflow version:      {mlflow.__version__}")
print(f"onnxmltools version: {onnxmltools.__version__}")
print(f"XGBoost version:     {xgboost.__version__}")

# Idempotently seed the SQLite DB from pii_datagen so this notebook is
# self-contained: rerun it and you always get the same 10k customers.
with get_conn() as conn:
    init_schema(conn)
    seed_offers(conn)
    seed_customers(conn, n=10_000, seed=42)

print(f"SQLite DB at: {DB_PATH}")

In [ ]:
USERNAME = os.environ.get("PROJECT_OWNER", "local")
DATE = date.today()
EXPERIMENT_NAME = f"nba-customer-risk-{USERNAME}"
REGISTERED_MODEL_NAME = "nba-risk-onnx-xgboost"

mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# Pull only the columns needed for training.  The 14 fraud-shaped features
# (balances, lat/long, transaction_amount) used by the original notebook are
# intentionally left behind — the guardrail is now a customer-risk scorer
# keyed off the fields the app can look up from state["customer_record"] at
# inference time.
with get_conn() as conn:
    df = pd.read_sql_query(
        "SELECT age, annual_income, existing_debt, employment_status, risk_tier "
        "FROM customers",
        conn,
    )

# Binary label: 1 if HIGH risk, else 0.  Keeps the ONNX output shape at
# [-1, 2] so nba_app.py's response parsing (outputs[1]["data"][1] = P(class=1))
# is unchanged.
y = (df["risk_tier"] == "HIGH").astype(np.int64)

# Numeric block.
X_numeric = df[["age", "annual_income", "existing_debt"]].astype(np.float32)

# One-hot employment_status.  Iterate the fixed category list from pii_datagen
# so column order is deterministic across train time and inference time even
# if some categories are absent from a given sample.
X_onehot = pd.DataFrame(
    {
        f"emp_{status}": (df["employment_status"] == status).astype(np.float32)
        for status in _EMP_STATUSES
    }
)

X = pd.concat([X_numeric, X_onehot], axis=1)
FEATURE_COLUMNS = list(X.columns)

print(f"features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print(f"label distribution: {y.value_counts().to_dict()}")

In [ ]:
test_size = 0.3
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)
print(f"train={X_train.shape} test={X_test.shape}")

In [ ]:
with mlflow.start_run():
    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Recall:   {recall * 100:.2f}%")
    print(f"Test size: {test_size * 100:.0f}%")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("n_features", len(FEATURE_COLUMNS))
    mlflow.log_param("feature_columns", ",".join(FEATURE_COLUMNS))

    # ONNX: single "input" tensor of shape [-1, n_features] to match the
    # AI Inference contract.  Outputs remain the ONNX-classifier defaults:
    # "label" (INT64 [-1, 1]) and "probabilities" (FP32 [-1, 2]), so the
    # deployed endpoint schema is compatible with nba_app.py's response
    # parsing (outputs[1]["data"][1] = P(high_risk)).
    num_features = X_train.shape[1]
    initial_type = [("input", FloatTensorType([None, num_features]))]
    onnx_model = onnxmltools.convert_xgboost(
        model.get_booster(), initial_types=initial_type
    )

    model_signature = infer_signature(X_train, y_pred)
    mlflow.onnx.log_model(
        onnx_model,
        "nba-risk-onnx-xgboost",
        registered_model_name=REGISTERED_MODEL_NAME,
        signature=model_signature,
    )